# Getting started with Spark on the data lake

This notebook walks through the full cycle: **read** lake tables, **transform** them with Spark, **write** a new table and **query** the result from Trino too.

- Run one cell at a time with **Shift+Enter**.
- The Spark session is already connected to the cluster (`spark://spark-master:7077`) and to the `lakehouse` catalog (Polaris + SeaweedFS).
- Layers: `bronze` (raw data), `silver` (clean, current state), `gold` (ready for BI). Experiments go to the **`sandbox`** namespace, never to gold.
- When you are done, run the last cell (`spark.stop()`): while the session is open, it holds 2 cluster cores.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("notebook-getting-started").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")  # errors only in the notebook; internal warnings stay out
spark

## 1. What is in the lake
`spark.sql` takes SQL and returns a DataFrame. `.show()` prints the first rows.

In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show()
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

## 2. Read a table
`spark.table(...)` returns a DataFrame. Nothing is read until an action (`show`, `count`, `toPandas`...): Spark builds the plan and runs it on the cluster.

In [ ]:
sales = spark.table("lakehouse.gold.daily_sales")
sales.printSchema()
sales.orderBy(F.desc("order_date"), F.desc("revenue")).show(10)

## 3. Transform: average ticket by channel
Starting from **silver** (orders and items from CDC), computes revenue, orders and average ticket per sales channel, excluding cancelled orders.

In [ ]:
orders = spark.table("lakehouse.silver.erp_orders")
items = spark.table("lakehouse.silver.erp_orderitems")

ticket = (
    items.join(orders, "OrderId")
    .where(F.col("Status") != "CANCELLED")
    .withColumn("channel", F.coalesce("Channel", F.lit("unknown")))
    .groupBy("channel")
    .agg(
        F.countDistinct("OrderId").alias("orders"),
        F.sum(F.col("Quantity") * F.col("UnitPrice")).cast("decimal(16,2)").alias("revenue"),
    )
    .withColumn("avg_ticket", (F.col("revenue") / F.col("orders")).cast("decimal(12,2)"))
    .orderBy(F.desc("revenue"))
)
ticket.show()

## 4. Write to the lake
`writeTo(...).createOrReplace()` creates the Iceberg table (or replaces it atomically). It is visible right away to Trino and to other jobs.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.sandbox")
ticket.writeTo("lakehouse.sandbox.avg_ticket_by_channel").createOrReplace()
spark.table("lakehouse.sandbox.avg_ticket_by_channel").show()

## 5. The same table from Trino
What Spark writes, Trino reads, because both use the same catalog. Requires the `query` profile to be up (`docker compose --profile query up -d`).

In [ ]:
import trino

try:
    with trino.dbapi.connect(host="trino", port=8080, user="notebook", catalog="lakehouse") as connection:
        cursor = connection.cursor()
        cursor.execute("SELECT * FROM sandbox.avg_ticket_by_channel ORDER BY revenue DESC")
        for row in cursor.fetchall():
            print(row)
except Exception as error:
    print("Trino unavailable (is the query profile up?):", error)

## 6. Time travel
Every write to Iceberg creates a *snapshot*. You can read the table as it was at a point in the past.

In [ ]:
spark.sql("""
    SELECT committed_at, operation, snapshot_id
    FROM lakehouse.silver.erp_orders.snapshots
    ORDER BY committed_at
""").show(5, truncate=False)

# The 1st snapshot is the table creation (empty); the 2nd is the initial load from the ERP.
second = spark.sql(
    "SELECT committed_at AS t FROM lakehouse.silver.erp_orders.snapshots ORDER BY committed_at LIMIT 1 OFFSET 1"
).first().t
before = spark.sql(f"SELECT count(*) AS orders FROM lakehouse.silver.erp_orders TIMESTAMP AS OF '{second}'").first().orders
now = spark.table("lakehouse.silver.erp_orders").count()
print(f"orders at {second:%Y-%m-%d %H:%M} (initial load): {before}  |  now: {now}")

## 7. To pandas and a chart
Small results can go to pandas (`toPandas()`), which runs here in the notebook. **Careful:** bring only aggregated or filtered data; a whole large table does not fit in the notebook's memory.

In [ ]:
import matplotlib.pyplot as plt

by_day = (
    spark.table("lakehouse.gold.daily_sales")
    .groupBy("order_date").agg(F.sum("revenue").alias("revenue"))
    .orderBy("order_date")
    .toPandas()
)
by_day["revenue"] = by_day["revenue"].astype(float)
by_day.plot(x="order_date", y="revenue", kind="bar", legend=False, title="Revenue per day (BRL)")
plt.tight_layout()
plt.show()

## 8. Finish
Frees the cluster cores for the scheduled jobs and streaming.

In [ ]:
spark.stop()